# Splash dynamics on a cone surface



## Documentation and Imports


Created on 27-07-21

Author: Valentin Laplaud

Formulas and graphs describing the dynamics of a drop splashing on a cone

In [1]:
# plotting stuff
import matplotlib as mpl
mpl.use('TkAgg')
%matplotlib inline

COLOR = 'white'
COLOR2 = 'black'

mpl.rcParams['text.color'] = COLOR
mpl.rcParams['axes.labelcolor'] = COLOR
mpl.rcParams['xtick.color'] = COLOR
mpl.rcParams['ytick.color'] = COLOR
mpl.rcParams['axes.edgecolor'] = COLOR

mpl.rcParams["figure.facecolor"] = COLOR2
mpl.rcParams["axes.facecolor"] = COLOR2
mpl.rcParams["savefig.facecolor"] = COLOR2

import matplotlib.pyplot as plt
from cycler import cycler


# numbers handling
import numpy as np
from numpy import pi
import pandas as pd

# signal processing 
from scipy.signal import savgol_filter, correlate, correlation_lags
from scipy.interpolate import interp1d


# to hide known warnings
import warnings
warnings.filterwarnings("ignore")

# General system functions
import os
import shutil
import sys

import time

# my functions
sys.path.append('../PythonFuncs/')
import VallapFunc_DP as vf
import DropGeometryClasses as dgc
import DropGeometryFuncs as dgf
import ImpactSimulationFuncs as isf


## Drop impact characteristics

### Top view schematics

In [2]:
# Xc = 0
# Yc = 0
# Rc = 5

# Xd = 6
# Yd = 0
# Rd = 4

# r = vf.dist(Xd,Yd,Xc,Yc)


# tx = np.linspace(0,2*pi,100)

# fig,ax = plt.subplots(dpi=600)
# ax.set_aspect('equal', adjustable='box') 
# ax.axis('off')

# # Cone
# ax.plot(Xc,Yc,'ow',ms = 2)
# ax.plot(Xc + Rc*np.cos(tx),Yc + Rc*np.sin(tx), '-w')
# ax.plot([Xc, Xc + Rc*np.cos(tx[30])], [Yc, Yc + Rc*np.sin(tx[30])], '--w', lw = 0.5)
# ax.text(Xc + Rc*np.cos(tx[36])/2,Yc+ Rc*np.sin(tx[36])/2, 'Rc', c = 'w', size = Rc*2)


# # Drop
# ax.plot(Xd,Yd,'ob',ms = 2)
# ax.plot(Xd + Rd*np.cos(tx),Yd + Rd*np.sin(tx), '-b')
# ax.plot([Xd, Xd + Rd*np.cos(tx[90])], [Yd, Yd + Rd*np.sin(tx[90])], '--b', lw = 0.5)
# ax.text(Xd + Rd*np.cos(tx[93])/2,Yd + Rd*np.sin(tx[93])/2, 'Rd', c = 'b', size = Rd*2)

# # Geometry definition
# ax.plot([Xc, Xd], [Yc, Yd], '--r', lw = 0.5, zorder = -1)
# ax.text((Xc+Xd)/2,(Yc+Yd)/2+Rd/10,'r', c= 'r', size = 10);


### Adimensional numbers

In [3]:
# ### Parameters
# V = 5 # [m/s] drop impact speed
# L = 0.003 # [m] cone diameter
# rho = 1000 # [kg/m3] water density
# mu = 0.001 # [Pa.s] water viscosity
# gamma = 0.07 # [N/m] water surface tension
# g = 9.8 # [m/s²] gravity constant

# ### Adimensional numbers

# # Reynolds (viscosity vs. intertia)
# Re = rho*V*L/mu
# print('Reynolds = ' + str(Re))

# # Weber (capilarity vs. inertia)
# We = rho*L*V**2/gamma
# print('Weber = ' + str(We))

# # Froude (gravity vs. intertia)
# Fr = V/np.sqrt(g*L)
# print('Froude = ' + str(Fr))

### Plots of fall probability


In [4]:

# dropR = 3
# coneR = 4 # Cone radius
# Alpha = pi/4 # Cone vertical angle

# xcircle = coneR*np.cos(np.linspace(0,2*pi,100))
# ycircle = coneR*np.sin(np.linspace(0,2*pi,100))
# xgrid = np.arange(-(coneR+dropR), coneR+dropR, 0.1)
# ygrid = np.arange(-(coneR+dropR), coneR+dropR, 0.1)
# xx, yy = np.meshgrid(xgrid, ygrid)
# mask = np.sqrt(xx**2+yy**2)<(coneR+dropR)
# z = dgf.ProbaFall(np.sqrt(xx[mask]**2+yy[mask]**2),dropR,coneR)


# ### Drop-Cone distance
# rstep = (coneR+dropR)/98
# nstep = int(np.ceil((coneR+dropR)/rstep))
# r = np.linspace(rstep/2,coneR+dropR-3*rstep/2,nstep) # impact position
# Pr = dgf.ProbaFall(r,dropR,coneR)


# # probability of falling at a certain distance of the cone center
# fig10,ax10 = plt.subplots(dpi = 500)
# fig10.suptitle("Probability of falling at a distance 'r' from the center \n dropR = " + str(dropR) + " mm, coneR = " + str(coneR) + " mm. ")
# ax10.plot(r/(coneR+dropR)*100,Pr,'o', ms = 1)
# ax10.set_xlabel(' r  [% of (coneR+dropR)]');
# ax10.set_ylabel('Probability [%]');
# fig10.tight_layout()

# fig11, ax11 = plt.subplots(dpi = 500)
# ax11.set_aspect('equal', adjustable='box')
# ax11.set_xlabel('X')
# ax11.set_ylabel('Y')
# fig11.suptitle("Probability of falling at a distance 'r' from the center \n dropR = " + str(dropR) + " mm, coneR = " + str(coneR) + " mm. ")


# ax11.plot(xcircle,ycircle,'-w')
# sc11 = ax11.scatter(xx[mask],yy[mask],c=z,s=1,cmap='jet')
# fig11.colorbar(sc11, ax = ax11, label = 'proba (%)',shrink = 0.6)


## Simulations

In [5]:
savepath = r'D:\Users\laplaud\Desktop\PostDoc\Data\SplashCups\Model\Images'

### Drop mesh

In [6]:
# Rd = 3
# Rc = 3.5
# r = 2.4

# Ad = 0
# Alpha = np.pi/4
# Beta = 2*np.pi*(1-np.sin(Alpha))

# ### Circle config : removed sector to form a cone 
                
# T1 = np.pi-Beta/2
# T2 = np.pi+Beta/2

# if T1>T2:
#     Ts = np.linspace(np.mod(T1+np.pi,2*np.pi),np.mod(T2+np.pi,2*np.pi),20) - np.pi        
# else:                
#     Ts = np.linspace(T1,T2,20)

# sectorT = np.append(np.append([T1],Ts),[T2])
# sectorR = np.append(np.append([0],Rc/np.sin(Alpha)*np.ones(20)),[0])

# sectorX,sectorY = vf.ToCart(sectorT,sectorR,angle = 'rad')

# n = 30

# tx = np.linspace(0.001,2*np.pi,2*n)+np.pi

# Xs = np.linspace(-Rd,Rd,n) + r
# Ys = np.linspace(-Rd,Rd,n)

# meshX,meshY = np.meshgrid(Xs,Ys)

# meshBorderX = Rd*0.999*np.cos(tx)+r
# meshBorderY = Rd*0.999*np.sin(tx)

# inConeAndDrop = (np.sqrt(np.square(meshX)+np.square(meshY))<Rc) & (np.sqrt(np.square(meshX-r)+np.square(meshY))<Rd)

# inConeAndDropBorder = (np.sqrt(np.square(meshBorderX)+np.square(meshBorderY))<Rc) 

# meshXimpact = meshX[inConeAndDrop]
# meshYimpact = meshY[inConeAndDrop]
# meshBorderXimpact = meshBorderX[inConeAndDropBorder]
# meshBorderYimpact = meshBorderY[inConeAndDropBorder]


# meshHimpact = 2*np.sqrt(Rd**2-(meshXimpact-r)**2-meshYimpact**2)
    
# meshHimpact[np.isnan(meshHimpact)] = 0

# meshHborder = 2*np.sqrt(Rd**2-(meshBorderXimpact-r)**2-meshBorderYimpact**2)
    
# meshHborder[np.isnan(meshHborder)] = 0


# # fig, ax = plt.subplots(figsize = (8,6),dpi =200)

# # ax.plot(Rc*np.cos(tx),Rc*np.sin(tx),'g',label = 'Cone')
# # plt.plot(0,0,'g.')

# # ax.plot(Rd*np.cos(tx)+r,Rd*np.sin(tx),'b',label = 'Drop')
# # plt.plot(r,0,'b.')


# # ax.scatter(meshX,meshY,c='gray',label='Square mesh',s=5)

# # ax.scatter(meshBorderX,meshBorderY,c='w',s = 5, label = 'Border mesh',zorder=5)

# # ax.set_aspect('equal')
# # ax.set_xticks([])
# # ax.set_yticks([])
# # ax.set_xlim([-Rc-1,r+Rd+0.6])
# # ax.set_ylim([-Rc-0.5,Rc+1.5])
# # plt.legend(fontsize='small',loc='upper left')


# # fig, ax = plt.subplots(figsize = (8,6),dpi =200)

# # ax.plot(Rc*np.cos(tx),Rc*np.sin(tx),'g',label = 'Cone')
# # plt.plot(0,0,'g.')

# # ax.plot(Rd*np.cos(tx)+r,Rd*np.sin(tx),'b',label = 'Drop')
# # plt.plot(r,0,'b.')


# # ax.scatter(meshX,meshY,c='gray',label='Square mesh',s=5)

# # ax.scatter(meshBorderX,meshBorderY,c='w',s = 5, label = 'Border mesh',zorder=5)


# # ax.scatter(meshXimpact,meshYimpact,c='r',label='Trajectories starting points',s=5,zorder=6)

# # ax.scatter(meshBorderXimpact,meshBorderYimpact,c='r',s = 5, label = None,zorder=6)

# # ax.set_aspect('equal')
# # ax.set_xticks([])
# # ax.set_yticks([])
# # ax.set_xlim([-Rc-1,r+Rd+0.6])
# # ax.set_ylim([-Rc-0.5,Rc+1.5])
# # plt.legend(fontsize='small',loc='upper left')


# # cmap = plt.get_cmap('Blues')
# # bluemap = vf.truncate_colormap(cmap,0.5,1,100)


# # fig, ax = plt.subplots(figsize = (8,6),dpi =200)

# # ax.plot(Rc*np.cos(tx),Rc*np.sin(tx),'g',label = 'Cone')
# # plt.plot(0,0,'g.')

# # ax.plot(Rd*np.cos(tx)+r,Rd*np.sin(tx),'b',label = 'Drop')
# # plt.plot(r,0,'b.')


# # q0 = ax.scatter(meshXimpact,meshYimpact,c=meshHimpact,cmap=bluemap,vmin=0, vmax = 6,label='Drop heigth on mesh',s=5,zorder=6)
# # ax.scatter(meshBorderXimpact,meshBorderYimpact,c=meshHborder,cmap=bluemap,vmin=0, vmax = 6,s = 5, label = None,zorder=6)

# # ax.set_aspect('equal')
# # ax.set_xticks([])
# # ax.set_yticks([])
# # ax.set_xlim([-Rc-1,r+Rd+0.6])
# # ax.set_ylim([-Rc-0.5,Rc+1.5])
# # plt.legend(fontsize='small',loc='upper left')

# # fig.colorbar(q0, ax = ax,orientation='horizontal',label = 'Drop height (mm)',fraction=0.046, pad=0.04)




# coneX,coneY = Rc/np.sin(Alpha)*np.cos(tx),Rc/np.sin(Alpha)*np.sin(tx)
# dropX,dropY = dgf.Cone2Circle(Rd*np.cos(tx)+r,Rd*np.sin(tx),Alpha,Ad)
# meshXimpact2,meshYimpact2 = dgf.Cone2Circle(meshXimpact,meshYimpact,Alpha,Ad)
# meshBorderXimpact2,meshBorderYimpact2 = dgf.Cone2Circle(meshBorderXimpact,meshBorderYimpact,Alpha,Ad)

# fig, ax = plt.subplots(figsize = (8,6),dpi =200)

# ax.plot(coneX,coneY,'g',label = 'Cone')
# plt.plot(0,0,'g.')
# ax.plot(sectorX,sectorY,'r--',label = 'Removed sector')
# plt.plot(0,0,'g.')

# ax.plot(dropX,dropY,'b',label = 'Drop')
# plt.plot(r,0,'b.')


# q0 = ax.scatter(meshXimpact2,meshYimpact2,c=meshHimpact,cmap=bluemap,vmin=0, vmax = 6,label='Drop heigth on mesh',s=5,zorder=6)
# ax.scatter(meshBorderXimpact2,meshBorderYimpact2,c=meshHborder,cmap=bluemap,vmin=0, vmax = 6,s = 5, label = None,zorder=6)

# ax.set_aspect('equal')
# ax.set_xticks([])
# ax.set_yticks([])
# plt.legend(fontsize='small',loc='upper left')

# fig.colorbar(q0, ax = ax,orientation='horizontal',label = 'Drop height (mm)',fraction=0.046, pad=0.04)


### Defining drops and cones

In [7]:
# ## Definig diverse drops and cones

# npts = 30

# DropDiam = 3

# D0 = dgc.Drop(DropDiam,0.01,npts,4)
# D1 = dgc.Drop(DropDiam,1,npts,4)
# D2 = dgc.Drop(DropDiam,2,npts,4)
# D3 = dgc.Drop(DropDiam,3,npts,4)
# D4 = dgc.Drop(DropDiam,4,npts,4)
# D5 = dgc.Drop(DropDiam,5,npts,4)
# D6 = dgc.Drop(DropDiam,6,npts,4)
# D7 = dgc.Drop(DropDiam,7,npts,4)

# Ds= [D0,D1,D2,D3,D4,D5]

# P = dgc.Cone(2.7,np.pi/2) # flat pillar

# # Experimental cones
# C45_27 = dgc.Cone(2.7,np.pi/4) 
# C45_35 = dgc.Cone(3.5,np.pi/4) 
# C45_54 = dgc.Cone(5.4,np.pi/4) 

# C60 = dgc.Cone(2.7,np.pi/3)
# C30 = dgc.Cone(2.7,np.pi/6)


# C20 = dgc.Cone(2.7,np.pi/9)



### Computing impacts

In [8]:


# I_D0_C35 = C45_35.impact(D0)
# I_D4_C45 = C45_27.impact(D2)


# I_D4_P = P.impact(D4)
# I_D1_C30 = C30.impact(D1)
# I_D1_C60 = C60.impact(D1)


# I_D2_C45 = C45_27.impact(D2)
# I_D3_C45 = C45_27.impact(D3)
# I_D4_C45 = C45_27.impact(D4)
# I_D5_C45 = C45_27.impact(D5)
# I_D6_C45 = C45_27.impact(D6)
# I_D7_C45 = C45_27.impact(D7)


### Plots

#### Impact visualizations

In [9]:

# npts = 31
# ConeDiam = 2.7
# DropDiam = ConeDiam*1.1

# Rocs = np.round(np.linspace(0.5,1.8,3)*10)/10
# Roc = 1.2

# D = dgc.Drop(DropDiam/2,Roc*ConeDiam/2,npts,5)
# C = dgc.Cone(ConeDiam/2,np.pi/4)
# C = dgc.Cone(ConeDiam/2,np.pi/2)
        
# start_time = time.time()

# oriType = 'Hmax'
# velIni = 'VelNorm'
# meshType = 'zone'

# title = oriType + '_' + velIni + '_' + meshType

# I = C.impact(D,oriType,velIni,meshType)
# # I2 = C.impact(D,'Drop','Radial','zone')



# I.plot_splash_init(veltype='full_div0',nolabels=True,title=title)
# I.plot_splash_init(veltype='norm',nolabels=True,title=title)

# print("\nTotal time : %.04f milliseconds" % ((time.time() - start_time)*1000))

# I.plot_splash_traj(np.linspace(0,1,20),'full_div0',nolabels=True,title=title)
# # I2.plot_splash_traj(np.linspace(0,1,20),'full_div0',nolabels=True)
    



#### Quantifications

##### JetFracs

In [10]:
# Angle = np.pi/6 # [rad]
# npts = 20
# DropDiam = 3 # Diameter in [mm]
# ConeDiams = np.linspace(2,6,5) # Diameter in [mm]
# # ConeDiams = [3] # Diameter in [mm]
# oriType = 'Central'
# velType = 'full_div0'


# isf.plotFracs(Angle,npts,DropDiam,ConeDiams,oriType,velType)

##### Phase Diagram

In [11]:
# oriType = 'Hgrad' # ,'Hmax','Central','CentralHeight','Drop'

# velType = 'full_div0'

# velIni = 'Radial' # 'VelNorm',

# meshType = 'zone'

# npts = 71 # XY resolution for the diagram

# OffCmax = 3 # [mm]

# marchantias = ['polymorpha'] #,'polymorpha','globosa'

# for marchantia in marchantias:
#     if marchantia == 'polymorpha':
#         ConeSize = 7.3 # [mm²] polymorpha :7.3 globosa : 12.1
#         RelOffCmax = OffCmax/np.sqrt(ConeSize/np.pi)
#         ConeSizeType = 'surface'
#     elif marchantia == 'globosa':
#         ConeSize = 12.1 # [mm²] polymorpha :7.3 globosa : 12.1
#         RelOffCmax = OffCmax/np.sqrt(ConeSize/np.pi)
#         ConeSizeType = 'surface'
#     elif marchantia == 'radius':
#         ConeSize = 3 # [mm] 
#         RelOffCmax = OffCmax
#         ConeSizeType = 'radius'
#     # 

#     # parameter space
#     RelDropDiams = np.linspace(0.125,RelOffCmax-1,npts)

#     RelOffCents = np.linspace(0.125,RelOffCmax,npts)

#     Angles = np.linspace(0.5,89.5,npts)/360*2*np.pi

#     label = velIni + '_' + oriType + '_' + velType + '_' + meshType + '_' + marchantia

#     print('\n\nRunning for : ' + label + '\n')

#     start_time = time.time()

#     isf.PhaseDiagrams(RelOffCents,ConeSize,ConeSizeType,Angles,RelDropDiams,oriType,velType,velIni,meshType,label)


##### Optimization diagram

In [ ]:
coneSurface = 7.3 # [mm²] polymorpha :7.3 globosa : 12.1

npts = 31 # resolution for the diagram

coneAngles = np.linspace(0.1,90,npts)/360*2*np.pi

ndrops = 2000
areaScaling = np.linspace(0.2,20,npts)
dropScaling = np.sqrt(areaScaling)

label = 'FullScale'

isf.OptiDiagrams(coneSurface,coneAngles,npts,ndrops,dropScaling,label)   

### Ana's experimental data

In [ ]:
# data = pd.read_csv(r'd:\Users\laplaud\Desktop\PostDoc\Data\SplashCups\DataAna\all_angles.csv')
# data['OffC_CD'] = data['OffCentering']/data['TargetDiam']*2
# data['DD_CD'] = data['DropDiam']/data['TargetDiam']
# data.loc[data['SplashType']=='Bell','color'] = 'r'
# data.loc[data['SplashType']=='Crown','color'] = 'r'
# data.loc[data['SplashType']=='Jet','color'] = 'b'
# data.loc[data['SplashType']=='Transition','color'] = 'g'
# data


In [ ]:
# data.plot.scatter('OffC_CD','DD_CD',c = 'color',legend=1)
# plt.xlim(0,1.5)
# plt.ylim(0,1.5)
# plt.xlabel('Offcentering/ConeSize')
# plt.ylabel('DropSize/ConeSize')
# # plt.gca().set_aspect('equal')

In [ ]:
# data = data.loc[(data['angle[degrees]']==45) &( data['IdealDiam[mm]']==2.7)]
# data

In [ ]:
# Rds = data['DropDiam'].values/2
# # Rcs = data['TargetDiam'].values
# OffCs = data['OffCentering'].values
# Names = data['Ind.1']

In [ ]:
# Drops = [dgc.Drop(Rd,OffC,35,5) for Rd,OffC in zip(Rds,OffCs)]
# C45 = dgc.Cone(2.7/2,np.pi/4)
# Impacts = [C45.impact(D) for D in Drops]

In [ ]:
# C45.impact(dgc.Drop(1.8,0.315,60,5)).plot_splash_traj(np.linspace(0,10,51),nolabels=True)

In [ ]:
# for t,i in zip(np.linspace(0,10,51),np.linspace(0,50,51)):
#     f,a = C45.impact(dgc.Drop(1.8,0.315,300,5)).plot_splash_traj([t],nolabels=True)
    
#     f.savefig(r'd:\Users\laplaud\Desktop\PostDoc\Code\DropProject_WithAna\TestFilm2\\' + str(i) + '.png' )
#     plt.close(f)

## Test Zone

### Difference between data

In [ ]:
# # Path
# path = r'd:\Users\laplaud\Desktop\PostDoc\Code\DropProject_WithAna\Figures'
# # Parameters


# npts = 21 # XY resolution for the diagram

# data = 'JetFracs'
# label = 'Jet fraction (%)'
# marktype = 's'

# colormap = 'PuOr' # 'PuOr' 'cividis' 'plasma' 'jet' 'rainbow'

# pointSize =17000/npts**2


# ConeDiam = 2.7 # Diameter in [mm]
# # 
# OffCmax = 3 # 

# DiagDim = 2

# # parameter space
# RelDropDiams = np.linspace(0.25,OffCmax-1,npts)

# RelOffCents = np.linspace(0.25,OffCmax,npts)

# Angles = np.linspace(20,70,npts)/360*2*np.pi

# pointSize = 16000/npts**2

# DD = RelDropDiams*ConeDiam

# OC = RelOffCents*ConeDiam/2

# meshA,meshDD,meshOC = np.meshgrid(Angles,DD,OC)    


# oriTypes = ['CentralHeight','Central','Drop','Hmax','Hgrad']

# velTypes = ['full_div0']

# velInis = ['Radial','VelNorm'] 

# # Condition 1
# i1 = 4 # 'CentralHeight','Central','Drop','Hmax','Hgrad'
# j1 = 0 # 'full_div0','full'
# k1 = 0 # 'Radial','VelNorm'
# origin1 = oriTypes[i1]
# veltype1 = velTypes[j1]
# velIni1 = velInis[k1]

# folder1 = velIni1 + '_' + origin1 + '_' + veltype1 + '_zone_' + str(npts) + 'npts'
# title1 = origin1

# loadpath1 = path + '\\' + folder1

# # Condition 2
# i2 = 2 # 'CentralHeight','Central','Drop','Hmax','Hgrad'
# j2 = 0 # 'full_div0','full'
# k2 = 1 # 'Radial','VelNorm'
# origin2 = oriTypes[i2]
# veltype2 = velTypes[j2]
# velIni2 = velInis[k2]

# folder2 = velIni2 + '_' + origin2 + '_' + veltype2 + '_zone_' + str(npts) + 'npts'
# title2 = origin2

# loadpath2 = path + '\\' + folder2

# Data1 = np.load(loadpath1 + '\Data_' + data + '_' + str(npts) + 'npts.npy')

# Data2 = np.load(loadpath2 + '\Data_' + data + '_' + str(npts) + 'npts.npy')


# plotAngle = 45/360*2*np.pi
# angleidx = np.argwhere(Angles == plotAngle)[0]
# plotData1 = Data1[:,angleidx[0],:]
# plotData2 = Data2[:,angleidx[0],:]


# f,[[ax0,ax1],[ax2,ax3]] = plt.subplots(nrows = 2,ncols=2,dpi=250,figsize = (7,6)) 
# f.suptitle('45° angle, data differences')

# # f.suptitle('Average all angle')
# # plotData1 = Data1.mean(axis=1)
# # plotData2 = Data2.mean(axis=1)

# ax0.set_title(title1  + '\n' + velIni1 + '_' +veltype1 + ' (dat1)')
# ax0.set_ylabel('DropSize/ConeSize')

# sc0 = ax0.scatter(meshOC[:,0,:]*2/ConeDiam,meshDD[:,0,:]/ConeDiam,c=plotData1,
#                   cmap = colormap,vmax = 100,s=pointSize,marker=marktype)
# cbar0 = plt.colorbar(sc0)
# cbar0.set_label(label)

# ax1.set_title(title2   + '\n' +velIni2 + '_'+ veltype2 + ' (dat2)' )
# sc1 = ax1.scatter(meshOC[:,0,:]*2/ConeDiam,meshDD[:,0,:]/ConeDiam,c=plotData2,
#                   vmax = 100,cmap = colormap,s=pointSize,marker=marktype)
# cbar1 = plt.colorbar(sc1)
# cbar1.set_label(label)

# ax2.set_title('|dat1-dat2|' )
# ax2.set_xlabel('Offcent/ConeRadius')
# ax2.set_ylabel('DropSize/ConeSize')
# sc2 = ax2.scatter(meshOC[:,0,:]*2/ConeDiam,meshDD[:,0,:]/ConeDiam,c=np.abs(plotData1-plotData2),
#                   cmap = 'jet',marker=marktype,s=pointSize)
# cbar2 = plt.colorbar(sc2)

# ax3.set_title('100*2*|dat1-dat2|/(dat2+dat1)')
# ax3.set_xlabel('Offcent/ConeRadius')
# sc3 = ax3.scatter(meshOC[:,0,:]*2/ConeDiam,meshDD[:,0,:]/ConeDiam,
#                   c=200*np.abs(np.divide((plotData1-plotData2),(plotData1+plotData2)))
#                   ,cmap = 'jet',s=pointSize,marker=marktype)
# cbar3 = plt.colorbar(sc3)
# cbar3.set_label('% of variation')

# f.tight_layout()



### Data gradient

In [ ]:
# # Path
# path = r'd:\Users\laplaud\Desktop\PostDoc\Code\DropProject_WithAna\Figures'

# npts = 41 # XY resolution for the diagram

# OffCmax = 3 # [mm]

# marktype = 's'

# colormap = 'plasma' # 'PuOr' 'cividis' 'plasma' 'jet' 'rainbow'

# pointSize =17000/npts**2

# # plots param
# data = 'JetNRJ'
# label = 'Kinetic energy of the jet [mJ]'


# marchantias = ['radius','polymorpha'] #,'polymorpha','globosa'

# for marchantia in marchantias:
#     if marchantia == 'polymorpha':
#         ConeSize = 7.3 # [mm²] polymorpha :7.3 globosa : 12.1
#         RelOffCmax = OffCmax/np.sqrt(ConeSize/np.pi)
#         ConeSizeType = 'surface'
#     elif marchantia == 'globosa':
#         ConeSize = 12.1 # [mm²] polymorpha :7.3 globosa : 12.1
#         RelOffCmax = OffCmax/np.sqrt(ConeSize/np.pi)
#         ConeSizeType = 'surface'
#     elif marchantia == 'radius':
#         ConeSize = 3 # [mm] 
#         RelOffCmax = OffCmax
#         ConeSizeType = 'radius'
#     # 

#     # parameter space
#     RelDropDiams = np.linspace(0.125,RelOffCmax-1,npts)

#     RelOffCents = np.linspace(0.125,RelOffCmax,npts)

#     Angles = np.linspace(0.5,89.5,npts)/360*2*np.pi

    
#     if ConeSizeType == 'surface':        
        
#         Normalisation = np.sqrt(ConeSize/np.pi)
        
#         NormStr = '_Norm'
        
#     elif ConeSizeType == 'radius':   
        
#         Normalisation = ConeSize
#         NormStr = '/ConeSize'

#     DD = RelDropDiams*Normalisation
#     OC = RelOffCents*Normalisation/2
        
#     meshA,meshDD,meshOC = np.meshgrid(Angles,DD,OC)  

#     folder = 'Radial_Hgrad_full_div0_zone_' + marchantia + '_'+  str(npts) + 'npts'
#     title = 'Hgrad_Radius'

#     print('Working on ' + folder)

#     loadpath = path + '\\' + folder


#     Data = np.load(loadpath + '\Data_' + data + '_' + str(npts) + 'npts.npy')


#     plotAngle = 45/360*2*np.pi
#     angleidx = np.argwhere(Angles == plotAngle)[0]
#     idx = angleidx[0]
#     for idx in range(npts):
#         plotData = Data[:,idx,:]

#         a = Angles[idx]            

#         Grad_y, Grad_x = np.gradient(plotData)
#         Grad_norm = np.sqrt(np.square(Grad_x)+np.square(Grad_x))

#         f0,ax = plt.subplots(dpi = 150, figsize = (7,6))
#         ax.set_title(title + str(np.round(Angles[idx]/(2*np.pi)*3600)/10) + '°_' + data + '_Gradient' )
#         ax.set_aspect(1.5)
#         sc = ax.scatter(meshOC[:,0,:]*2/Normalisation,meshDD[:,0,:]/Normalisation,c=Grad_norm,
#                          marker=marktype,cmap = 'jet',s=120000/npts**2)

#         cbar = plt.colorbar(sc,ticks=[])
#         cbar.set_label(label)
#         cbar.set_label('Data gradient')

#         left,right = ax.get_xlim()
#         bot,top = ax.get_ylim()

#         x = np.linspace(bot,top,20)

#         ax.plot(x,x,'--r',lw=4,zorder = 3,label = 'OffC = DropSize')

#         ax.plot([1,1],[0,1],'-.r',lw=4,zorder = 3,label = 'OffC = 1') # V = 0

#         ax.set_xlim(left,right)
#         ax.set_ylim(bot,top)

#         f0.tight_layout()
#         os.makedirs(loadpath + '\Gradients\\' + data + 'Single\\',exist_ok=True) # create folder
#         f0.savefig(loadpath + '\Gradients\\' + data + 'Single\\' + '\Gradient_' + data + '_' + str(np.round(Angles[idx]/(2*np.pi)*3600)/10) + '°.png')
#         plt.close(f0)


#         f,[[ax0,ax1],[ax2,ax3]] = plt.subplots(nrows = 2,ncols=2,dpi=250,figsize = (7,6)) 
#         f.suptitle(str(np.round(Angles[idx]/(2*np.pi)*3600)/10) + '° angle, '+ data +' gradient')



#         ax0.set_title(title  +  '(data)')
#         ax0.set_ylabel('DropSize/ConeSize')
#         ax0.set_aspect(1.5)
#         sc0 = ax0.scatter(meshOC[:,0,:]*2/Normalisation,meshDD[:,0,:]/Normalisation,c=plotData,
#                           cmap = colormap,marker=marktype,
#                           s=pointSize)

#     #             left,right = ax0.get_xlim()
#     #             bot,top = ax0.get_ylim()

#     #             x = np.linspace(bot,top,20)

#     #             ax0.plot(1+x*(np.tan(a)-1),x,'--r',zorder = 3,label = 'W = 0')
#     #             ax0.plot(x*np.tan(a),x,'--m',zorder = 4,label = 'W = Rd - Rc')

#     #             ax0.plot(np.divide(x,x),x,'-r',zorder = 3,label = 'V = 0') # V = 0
#     #             ax0.plot(1 + np.tan(a)*(x-1),x,'-m',zorder = 4,label = 'V = Rd - Rc') # V = Rd - Rc

#     #             ax0.set_xlim(left,right)
#     #             ax0.set_ylim(bot,top)

#         cbar0 = plt.colorbar(sc0)
#         cbar0.set_label(label)


#         ax1.set_title(title   + ' (gradient norm)' )
#         ax1.set_aspect(1.5)
#         sc1 = ax1.scatter(meshOC[:,0,:]*2/Normalisation,meshDD[:,0,:]/Normalisation,c=Grad_norm,
#                          marker=marktype,cmap = 'jet',s=pointSize)

#     #             left,right = ax1.get_xlim()
#     #             bot,top = ax1.get_ylim()

#     #             x = np.linspace(bot,top,20)

#     #             ax1.plot(1+x*(np.tan(a)-1),x,'--r',zorder = 3,label = 'W = 0')
#     #             ax1.plot(x*np.tan(a),x,'--m',zorder = 4,label = 'W = Rd - Rc')

#     #             ax1.plot(np.divide(x,x),x,'-r',zorder = 3,label = 'V = 0') # V = 0
#     #             ax1.plot(1 + np.tan(a)*(x-1),x,'-m',zorder = 4,label = 'V = Rd - Rc') # V = Rd - Rc

#     #             ax1.set_xlim(left,right)
#     #             ax1.set_ylim(bot,top)

#         cbar1 = plt.colorbar(sc1)
#         cbar1.set_label(label)
#         cbar1.set_label('Gradnorm')

#         ax2.set_title(title  + ' (gradient X)')
#         ax2.set_aspect(1.5)
#         ax2.set_xlabel('Offcent/ConeRadius')
#         ax2.set_ylabel('DropSize/ConeSize')
#         sc2 = ax2.scatter(meshOC[:,0,:]*2/Normalisation,meshDD[:,0,:]/Normalisation,c=Grad_x,marker=marktype,
#                           cmap = 'rainbow',s=pointSize)
#     #             left,right = ax2.get_xlim()
#     #             bot,top = ax2.get_ylim()

#     #             x = np.linspace(bot,top,20)

#     #             ax2.plot(1+x*(np.tan(a)-1),x,'--r',zorder = 3,label = 'W = 0')
#     #             ax2.plot(x*np.tan(a),x,'--m',zorder = 4,label = 'W = Rd - Rc')

#     #             ax2.plot(np.divide(x,x),x,'-r',zorder = 3,label = 'V = 0') # V = 0
#     #             ax2.plot(1 + np.tan(a)*(x-1),x,'-m',zorder = 4,label = 'V = Rd - Rc') # V = Rd - Rc

#     #             ax2.set_xlim(left,right)
#     #             ax2.set_ylim(bot,top)

#         cbar2 = plt.colorbar(sc2)
#         cbar2.set_label('GradX')

#         ax3.set_title(title + ' (gradient Y)')
#         ax3.set_aspect(1.5)
#         ax3.set_xlabel('Offcent/ConeRadius')
#         sc3 = ax3.scatter(meshOC[:,0,:]*2/Normalisation,meshDD[:,0,:]/Normalisation,marker=marktype,
#                           c=-Grad_y,cmap = 'rainbow',s=pointSize)
#     #             left,right = ax3.get_xlim()
#     #             bot,top = ax3.get_ylim()

#     #             x = np.linspace(bot,top,20)

#     #             ax3.plot(1+x*(np.tan(a)-1),x,'--r',zorder = 3,label = 'W = 0')
#     #             ax3.plot(x*np.tan(a),x,'--m',zorder = 4,label = 'W = Rd - Rc')

#     #             ax3.plot(np.divide(x,x),x,'-r',zorder = 3,label = 'V = 0') # V = 0
#     #             ax3.plot(1 + np.tan(a)*(x-1),x,'-m',zorder = 4,label = 'V = Rd - Rc') # V = Rd - Rc



#     #             ax3.set_xlim(left,right)
#     #             ax3.set_ylim(bot,top)

#         cbar3 = plt.colorbar(sc3)
#         cbar3.set_label('-GradY')

#         f.tight_layout()
#         os.makedirs(loadpath + '\Gradients\\' + data + 'Detailed\\',exist_ok=True) # create folder
#         f.savefig(loadpath + '\Gradients\\' + data + 'Detailed\\' + '\Gradient_' + data + '_' + str(np.round(Angles[idx]/(2*np.pi)*3600)/10) + '°.png')
#         plt.close(f)

#     plotData = Data.mean(axis=1)

#     Grad_y, Grad_x = np.gradient(plotData)
#     Grad_norm = np.sqrt(np.square(Grad_x)+np.square(Grad_x))

#     f0,ax = plt.subplots(dpi = 150, figsize = (7,6))
#     ax.set_title(title   + str(np.round(Angles[idx]/(2*np.pi)*3600)/10) + '°_' + data + '_Gradient' )
#     ax.set_aspect(1.5)
#     sc = ax.scatter(meshOC[:,0,:]*2/Normalisation,meshDD[:,0,:]/Normalisation,c=Grad_norm,
#                      marker=marktype,cmap = 'jet',s=120000/npts**2)

#     cbar = plt.colorbar(sc,ticks=[])
#     cbar.set_label(label)
#     cbar.set_label('Data gradient')

#     f0.tight_layout()
#     os.makedirs(loadpath + '\Gradients\\' + data + 'Single\\',exist_ok=True) # create folder
#     f0.savefig(loadpath + '\Gradients\\' + data + 'Single\\' + '\Gradient_' + data + '_AverageAngle.png')
#     plt.close(f0)


#     f,[[ax0,ax1],[ax2,ax3]] = plt.subplots(nrows = 2,ncols=2,dpi=250,figsize = (7,6)) 
#     f.suptitle('averaged on angles, data gradient')

#     ax0.set_title(title  + '(data)')
#     ax0.set_ylabel('DropSize/ConeSize')
#     ax0.set_aspect(1.5)
#     sc0 = ax0.scatter(meshOC[:,0,:]*2/Normalisation,meshDD[:,0,:]/Normalisation,c=plotData,
#                       cmap = colormap,marker=marktype,
#                       s=pointSize)

#     cbar0 = plt.colorbar(sc0)
#     cbar0.set_label(label)


#     ax1.set_title(title   +' (gradient norm)' )
#     ax1.set_aspect(1.5)
#     sc1 = ax1.scatter(meshOC[:,0,:]*2/Normalisation,meshDD[:,0,:]/Normalisation,c=Grad_norm,
#                      marker=marktype,cmap = 'jet',s=pointSize)
#     cbar1 = plt.colorbar(sc1)
#     cbar1.set_label(label)
#     cbar1.set_label('Gradnorm')

#     ax2.set_title(title   + ' (gradient X)')
#     ax2.set_aspect(1.5)
#     ax2.set_xlabel('Offcent/ConeRadius')
#     ax2.set_ylabel('DropSize/ConeSize')
#     sc2 = ax2.scatter(meshOC[:,0,:]*2/Normalisation,meshDD[:,0,:]/Normalisation,c=Grad_x,marker=marktype,
#                       cmap = 'rainbow',s=pointSize)
#     cbar2 = plt.colorbar(sc2)
#     cbar2.set_label('GradX')

#     ax3.set_title(title   + ' (gradient Y)')
#     ax3.set_aspect(1.5)
#     ax3.set_xlabel('Offcent/ConeRadius')
#     sc3 = ax3.scatter(meshOC[:,0,:]*2/Normalisation,meshDD[:,0,:]/Normalisation,marker=marktype,
#                       c=-Grad_y,cmap = 'rainbow',s=pointSize)
#     cbar3 = plt.colorbar(sc3)
#     cbar3.set_label('-GradY')

#     f.tight_layout()
#     os.makedirs(loadpath + '\Gradients\\' + data + 'Detailed\\',exist_ok=True) # create folder
#     f.savefig(loadpath + '\Gradients\\' + data + 'Detailed\\' + '\Gradient_' + data + '_AngleAverage.png')
#     plt.close(f)

#     print('Done.')